In [1]:
import sys
from pathlib import Path
import pandas as pd

RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ))   # para poder importar desde src/

from src.features import agregar_variables, BASE, DERIVADAS

df = pd.read_csv(RAIZ / "data" / "processed" / "concreto_limpio.csv")
df_f = agregar_variables(df)

print(df_f.shape)
print("Infinitos o NaN:", df_f[DERIVADAS].isna().sum().sum())
df_f[DERIVADAS].describe().T

(1005, 16)
Infinitos o NaN: 0


,count,mean,std,min,25%,50%,75%,max
ligantes,1005.0,406.211144,91.421010,200.000000,336.300000,388.400000,480.000000,640.000000
a_lig,1005.0,0.473136,0.125497,0.235073,0.389522,0.480000,0.561298,0.900000
a_c,1005.0,0.756220,0.313524,0.266893,0.547465,0.689531,0.937241,1.882353
frac_adiciones,1005.0,0.310665,0.211445,0.000000,0.187614,0.320767,0.465327,0.736000
frac_arena,1005.0,0.441987,0.035010,0.347841,0.424062,0.437904,0.471367,0.538082
log_edad,1005.0,3.168492,1.189466,0.000000,1.945910,3.332205,4.025352,5.899897


In [2]:
df_f[DERIVADAS + ["resistencia"]].corr(method="spearman")["resistencia"].drop("resistencia").sort_values()

a_lig            -0.590790
a_c              -0.503996
frac_adiciones   -0.140773
frac_arena       -0.024964
ligantes          0.584315
log_edad          0.605411
Name: resistencia, dtype: float64

In [3]:
df_f.to_csv(RAIZ / "data" / "processed" / "concreto_features.csv", index=False)

In [4]:
df_f["grupo_edad"] = pd.cut(df_f["edad"], [0, 7, 28, 400],
                            labels=["≤7 d", "8–28 d", ">28 d"])
for g, sub in df_f.groupby("grupo_edad", observed=True):
    r = sub["frac_adiciones"].corr(sub["resistencia"], method="spearman")
    print(f"{g}: ρ(frac_adiciones, resistencia) = {r:+.2f}  (n={len(sub)})")

≤7 d: ρ(frac_adiciones, resistencia) = -0.24  (n=253)
8–28 d: ρ(frac_adiciones, resistencia) = -0.25  (n=481)
>28 d: ρ(frac_adiciones, resistencia) = +0.04  (n=271)


- La correlación de frac_adiciones con la resistencia es -0.24 (≤7 d), -0.25 (8-28 d) y +0.04 (>28 d):
  la penalización temprana de escoria y ceniza se diluye con la edad.
  No se observa una ventaja a edades largas en esta correlación marginal; se revisará con SHAP.